In [1]:
import kagglehub
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import os

# Define una ruta manual para guardar los recursos de nltk
nltk_data_path = os.path.expanduser("~/nltk_data")

# Crea el directorio si no existe
os.makedirs(nltk_data_path, exist_ok=True)

# Descarga los recursos a esa ruta
nltk.download('punkt', download_dir=nltk_data_path)
nltk.download('stopwords', download_dir=nltk_data_path)
nltk.download('wordnet', download_dir=nltk_data_path)
nltk.download('omw-1.4', download_dir=nltk_data_path)

# Establece la ruta para que nltk busque ahí
nltk.data.path.append(nltk_data_path)





path = kagglehub.dataset_download("jp797498e/twitter-entity-sentiment-analysis")
print("Ruta al dataset:", path)


# Ruta al dataset


# Cargar CSV (sin encabezados)
df = pd.read_csv('twitter_training.csv') # Cambia el separador si es necesario


# Asignar nombres de columnas
df.columns = ['id', 'entity', 'sentiment', 'tweet']

# Mostrar primeras filas
df.head()

# Verificar columnas y tipos
df.info()

# Contar valores de sentimiento
print("Distribución de sentimientos:")
print(df['sentiment'].unique())


#Vasualizar sentimientos
sns.set(style="whitegrid")
plt.figure(figsize=(6,4))
sns.countplot(x='sentiment', data=df, palette='Set2')
plt.title('Distribución de sentimientos en los tuits')
plt.xlabel('Sentimiento')
plt.ylabel('Cantidad')
plt.show()


nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

from wordcloud import WordCloud
import string

def generar_wordcloud(sentimiento):
    # Verifica que el sentimiento esté en los datos
    if sentimiento not in df['sentiment'].unique():
        print(f"Sentimiento '{sentimiento}' no encontrado en los datos.")
        return

    # Filtra los tuits del sentimiento solicitado
    text = " ".join(tweet for tweet in df[df['sentiment'] == sentimiento]['tweet'].dropna())
    
    if not text:
        print(f"No hay tuits para el sentimiento '{sentimiento}'.")
        return

    # Generar WordCloud
    text = text.lower().translate(str.maketrans('', '', string.punctuation))
    wordcloud = WordCloud(stopwords=stop_words, background_color='white', width=800, height=400).generate(text)

    # Mostrar WordCloud
    plt.figure(figsize=(10,5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis("off")
    plt.title(f"Palabras más frecuentes en tuits '{sentimiento}'")
    plt.show()

# Ahora prueba generar la wordcloud para los tuits positivos
generar_wordcloud('Positive')

    



# Ahora ejecuta la función con un sentimiento válido


# Generar WordClouds
generar_wordcloud('Positive')
generar_wordcloud('Negative')
generar_wordcloud('Neutral')



#Prediccion

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def limpiar_texto_basica(texto):
    texto = str(texto).lower()
    texto = re.sub(r"http\S+|www\S+|https\S+", '', texto)
    texto = re.sub(r'\@w+|\#','', texto)
    texto = re.sub(r'[^A-Za-z\s]', '', texto)
    palabras = texto.split()
    palabras = [p for p in palabras if p not in stop_words]
    return " ".join(palabras)


# Hay valores nulos
print("Nulos en 'tweet':", df['tweet'].isnull().sum())

# Tipos únicos de datos
print("Tipos únicos en 'tweet':", df['tweet'].map(type).value_counts())



# Convierte todo a string y elimina nulos
df['tweet'] = df['tweet'].astype(str)
df = df[df['tweet'].notnull()]



# Aplicar limpieza


# Descarga todos los recursos necesarios (solo se hace una vez)
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')



df['clean_tweet'] = df['tweet'].apply(limpiar_texto_basica)

df[['tweet', 'clean_tweet']].head()


df['viral'] = df['sentiment'].apply(lambda x: 1 if x == 'Positive' else 0)
df[['sentiment', 'viral']].head()

vectorizer = TfidfVectorizer(max_features=1000)
X = vectorizer.fit_transform(df['clean_tweet'])
y = df['viral']


# Dividir datos
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Entrenar modelo
model = LogisticRegression()
model.fit(X_train, y_train)

# Evaluar
y_pred = model.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))



# Si no existen, agrega esta línea para simular:
df['retweets'] = np.random.randint(0, 200, size=len(df))

# Definir si es viral
df['viral'] = (df['retweets'] > 100).astype(int)

# Vectorizar texto
vectorizador = TfidfVectorizer(max_features=3000)
X = vectorizador.fit_transform(df['clean_tweet'])

y = df['viral']

# División entrenamiento/prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Entrenamiento
modelo = LogisticRegression()
modelo.fit(X_train, y_train)

# Evaluación
y_pred = modelo.predict(X_test)
print(classification_report(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred))

c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package punkt to C:\Users\User/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\User/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to C:\Users\User/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Ruta al dataset: C:\Users\User\.cache\kagglehub\datasets\jp797498e\twitter-entity-sentiment-analysis\versions\2


FileNotFoundError: [Errno 2] No such file or directory: 'twitter_training.csv'